In [86]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from io import StringIO

In [87]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
response.raise_for_status()

sp500_table = pd.read_html(StringIO(response.text))[0]

print("Wikipedia table loaded.")
print("Number of rows in S&P 500 table:", len(sp500_table))
sp500_table.head()

Wikipedia table loaded.
Number of rows in S&P 500 table: 503


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [88]:
tickers_raw = sp500_table["Symbol"].tolist()[:410]

# Convert BRK.B -> BRK-B etc. for yfinance
tickers = [t.replace(".", "-") for t in tickers_raw]

print("Number of raw tickers pulled:", len(tickers))
print("First 20 tickers:")
print(tickers[:20])
print("Last 10 tickers in buffer:")
print(tickers[-10:])

Number of raw tickers pulled: 410
First 20 tickers:
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL']
Last 10 tickers in buffer:
['ROP', 'ROST', 'RCL', 'SPGI', 'CRM', 'SNDK', 'SBAC', 'SLB', 'STX', 'SRE']


In [89]:
start_date = "2025-05-02"
end_date   = "2025-10-31"

print("Start date:", start_date)
print("End date:", end_date)

Start date: 2025-05-02
End date: 2025-10-31


In [90]:
prices = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    interval="1wk",
    auto_adjust=True,
    progress=True
)

print("Download finished.")

[*********************100%***********************]  410 of 410 completed

Download finished.


In [91]:
weekly_prices = prices["Close"].copy()

print("Raw weekly price table shape:")
print(weekly_prices.shape)

weekly_prices.head()

Raw weekly price table shape:
(27, 410)


Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,SNDK,SPGI,SRE,STX,STZ,T,TAP,TECH,XOM,XYZ
Date,,,,,,,,,,,,,,,,,,,,,
2025-04-28,107.991463,204.458893,193.802795,125.260002,130.977646,92.480003,303.810516,380.869995,195.605652,46.219826,...,34.400002,502.639893,73.236275,92.046013,182.238907,26.732265,55.348602,50.197235,102.681923,46.529999
2025-05-05,106.301460,197.668488,180.258972,127.040001,131.391815,94.540001,306.367737,383.279999,204.391617,46.876659,...,37.740002,503.741180,73.187767,94.656967,186.283417,26.954954,52.531170,49.650421,103.745377,50.360001
2025-05-12,113.101250,210.343246,179.692612,138.000000,132.930161,94.589996,316.109039,417.130005,223.697067,48.422153,...,40.439999,518.355469,75.312950,106.604057,191.722824,26.838766,54.665874,48.865002,104.596153,57.660000
2025-05-19,107.892044,194.677567,178.950485,126.720001,129.478714,92.070000,308.039368,407.690002,207.307144,46.306759,...,37.279999,506.013153,75.390587,111.499588,180.745972,26.548304,51.934990,46.508743,100.524956,58.740002
2025-05-26,111.262115,200.240662,181.733459,129.000000,131.727081,95.040001,315.243378,415.089996,210.764374,47.104450,...,37.689999,508.830841,76.263939,116.642372,174.738083,26.916222,51.531124,48.197807,99.812714,61.750000


In [92]:
missing_counts = weekly_prices.isna().sum()
bad_tickers = missing_counts[missing_counts > 0].sort_values(ascending=False)

print("Number of tickers with missing values:", len(bad_tickers))
print("\nBad tickers:")
print(bad_tickers)

Number of tickers with missing values: 1

Bad tickers:
Ticker
Q    26
dtype: int64


In [93]:
good_tickers = missing_counts[missing_counts == 0].index.tolist()

# Keep exactly the first 400 usable stocks
good_tickers_400 = good_tickers[:400]

weekly_prices_clean = weekly_prices[good_tickers_400].copy()

print("Number of usable tickers found:", len(good_tickers))
print("Final clean shape:", weekly_prices_clean.shape)
print("Final number of stocks used:", len(good_tickers_400))

Number of usable tickers found: 409
Final clean shape: (27, 400)
Final number of stocks used: 400


In [94]:
print("First 20 final tickers:")
print(good_tickers_400[:20])

print("\nLast 10 final tickers:")
print(good_tickers_400[-10:])

First 20 final tickers:
['A', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'AEE', 'AEP', 'AES', 'AFL', 'AIG', 'AIZ', 'AJG', 'AKAM']

Last 10 final tickers:
['ROP', 'ROST', 'RSG', 'RTX', 'RVTY', 'SATS', 'SBAC', 'SCHW', 'SLB', 'SNDK']


In [95]:
weekly_prices_clean.to_excel("step1_weekly_prices_clean_400.xlsx")
print("Saved as step1_weekly_prices_400.xlsx")

Saved as step1_weekly_prices_400.xlsx


In [96]:
# Compute weekly simple returns for the 400 stocks
stock_returns = weekly_prices_clean.pct_change().iloc[1:].copy()

print("Shape of weekly stock returns:")
print(stock_returns.shape)

stock_returns.head()

Shape of weekly stock returns:
(26, 400)


Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,ROP,ROST,RSG,RTX,RVTY,SATS,SBAC,SCHW,SLB,SNDK
Date,,,,,,,,,,,,,,,,,,,,,
2025-05-05,-0.015649,-0.033212,-0.069885,0.014210,0.003162,0.022275,0.008417,0.006328,0.044917,0.014211,...,0.007844,0.012741,-0.007744,-0.011827,-0.029152,0.006658,-0.009382,0.016364,-0.005183,0.097093
2025-05-12,0.063967,0.064121,-0.003142,0.086272,0.011708,0.000529,0.031796,0.088317,0.094453,0.032969,...,0.021736,0.074782,-0.003178,0.052926,0.022027,-0.033072,-0.027577,0.053329,0.035311,0.071542
2025-05-19,-0.046058,-0.074477,-0.004130,-0.081739,-0.025964,-0.026641,-0.025528,-0.022631,-0.073268,-0.043687,...,-0.027192,-0.102406,0.020461,-0.023103,-0.042140,-0.155194,-0.015274,-0.014545,-0.058988,-0.078140
2025-05-26,0.031236,0.028576,0.015552,0.017992,0.017365,0.032258,0.023387,0.018151,0.016677,0.017226,...,0.005714,0.020618,0.017520,0.036462,0.012202,-0.102733,0.015185,0.010755,-0.018122,0.010998
2025-06-02,0.034668,0.015285,0.019988,0.090233,0.000000,0.005156,0.002620,0.004409,0.038695,-0.017609,...,0.004559,0.026483,-0.016790,0.019197,0.017142,-0.014100,-0.026780,-0.001019,0.024811,0.038737


In [97]:
# Align correctly
stock_returns, rf_returns = stock_returns.align(rf_returns, join='inner', axis=0)

print("Aligned shapes:")
print("Stocks:", stock_returns.shape)
print("RF:", rf_returns.shape)

Aligned shapes:
Stocks: (26, 400)
RF: (26, 1)


In [98]:
excess_returns = stock_returns.sub(rf_returns.squeeze(), axis=0)

print("Excess returns shape:")
print(excess_returns.shape)

Excess returns shape:
(26, 400)


In [99]:
X = excess_returns.T.copy()

print("X shape:", X.shape)

X shape: (400, 26)


In [100]:
print("Total missing values in stock_returns:")
print(stock_returns.isna().sum().sum())

Total missing values in stock_returns:
0


In [103]:
# Make a protected copy (so you don't overwrite it later)
stock_returns_final = stock_returns.copy()

In [105]:
stock_returns_final

Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,ROP,ROST,RSG,RTX,RVTY,SATS,SBAC,SCHW,SLB,SNDK
Date,,,,,,,,,,,,,,,,,,,,,
2025-05-05,-0.015649,-0.033212,-0.069885,0.014210,0.003162,0.022275,0.008417,0.006328,0.044917,0.014211,...,0.007844,0.012741,-0.007744,-0.011827,-0.029152,0.006658,-0.009382,0.016364,-0.005183,0.097093
2025-05-12,0.063967,0.064121,-0.003142,0.086272,0.011708,0.000529,0.031796,0.088317,0.094453,0.032969,...,0.021736,0.074782,-0.003178,0.052926,0.022027,-0.033072,-0.027577,0.053329,0.035311,0.071542
2025-05-19,-0.046058,-0.074477,-0.004130,-0.081739,-0.025964,-0.026641,-0.025528,-0.022631,-0.073268,-0.043687,...,-0.027192,-0.102406,0.020461,-0.023103,-0.042140,-0.155194,-0.015274,-0.014545,-0.058988,-0.078140
2025-05-26,0.031236,0.028576,0.015552,0.017992,0.017365,0.032258,0.023387,0.018151,0.016677,0.017226,...,0.005714,0.020618,0.017520,0.036462,0.012202,-0.102733,0.015185,0.010755,-0.018122,0.010998
2025-06-02,0.034668,0.015285,0.019988,0.090233,0.000000,0.005156,0.002620,0.004409,0.038695,-0.017609,...,0.004559,0.026483,-0.016790,0.019197,0.017142,-0.014100,-0.026780,-0.001019,0.024811,0.038737
2025-06-09,0.011053,-0.036632,0.006585,-0.036263,0.015272,-0.047524,-0.018700,-0.060539,0.017073,0.096584,...,-0.021157,-0.083102,-0.006760,0.047376,0.016962,-0.036613,0.009527,-0.010085,0.080775,0.085568
2025-06-16,-0.012983,0.023161,-0.030249,-0.028110,-0.019392,-0.009012,-0.084502,-0.037684,0.014754,0.030962,...,0.002211,-0.028936,-0.012218,0.006521,0.015610,0.491686,0.006496,0.021177,-0.012672,0.096000
2025-06-23,0.031239,0.000398,-0.016136,0.021180,0.010452,0.001664,0.035358,0.023639,0.037705,-0.030405,...,0.002705,0.001802,-0.015391,-0.013502,0.029793,0.146895,0.006498,0.008631,-0.051060,0.012237
2025-06-30,0.018545,0.062015,0.038232,0.014645,0.000447,-0.012511,0.031544,-0.016899,0.036799,0.064063,...,0.016841,0.028538,-0.013218,0.007535,0.035678,0.088511,0.013561,0.016670,0.048221,-0.015695


In [106]:
stock_returns_final.to_excel("step2_stock_returns_400.xlsx")
print("Saved as step2_stock_returns_400.xlsx")

Saved as step2_stock_returns_400.xlsx


In [107]:
rf_data = yf.download(
    "BIL",
    start=start_date,
    end=end_date,
    interval="1wk",
    auto_adjust=True,
    progress=False
)

rf_prices = rf_data["Close"].copy()
rf_returns = rf_prices.pct_change().iloc[1:].copy()

print("Shape of rf_returns:")
print(rf_returns.shape)

rf_returns.head()

Shape of rf_returns:
(26, 1)


Ticker,BIL
Date,
2025-05-05,0.004181
2025-05-12,0.000765
2025-05-19,0.000983
2025-05-26,0.000545
2025-06-02,-0.002616


In [108]:
stock_returns_aligned, rf_returns_aligned = stock_returns_final.align(
    rf_returns,
    join="inner",
    axis=0
)

print("Aligned shapes:")
print("Stocks:", stock_returns_aligned.shape)
print("RF:", rf_returns_aligned.shape)

Aligned shapes:
Stocks: (26, 400)
RF: (26, 1)


In [109]:
excess_returns = stock_returns_aligned.sub(rf_returns_aligned.squeeze(), axis=0)

print("Shape of excess_returns:")
print(excess_returns.shape)

excess_returns.head()

Shape of excess_returns:
(26, 400)


Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,ROP,ROST,RSG,RTX,RVTY,SATS,SBAC,SCHW,SLB,SNDK
Date,,,,,,,,,,,,,,,,,,,,,
2025-05-05,-0.019831,-0.037393,-0.074066,0.010029,-0.001019,0.018094,0.004236,0.002146,0.040735,0.010030,...,0.003663,0.008560,-0.011925,-0.016008,-0.033333,0.002477,-0.013563,0.012183,-0.009364,0.092912
2025-05-12,0.063202,0.063356,-0.003907,0.085507,0.010943,-0.000236,0.031031,0.087552,0.093688,0.032204,...,0.020971,0.074017,-0.003943,0.052161,0.021262,-0.033836,-0.028342,0.052564,0.034546,0.070777
2025-05-19,-0.047040,-0.075459,-0.005112,-0.082722,-0.026947,-0.027624,-0.026511,-0.023613,-0.074251,-0.044669,...,-0.028175,-0.103389,0.019478,-0.024086,-0.043123,-0.156177,-0.016256,-0.015528,-0.059971,-0.079123
2025-05-26,0.030690,0.028031,0.015006,0.017447,0.016819,0.031713,0.022841,0.017606,0.016131,0.016681,...,0.005169,0.020072,0.016974,0.035917,0.011657,-0.103278,0.014640,0.010210,-0.018668,0.010452
2025-06-02,0.037284,0.017901,0.022604,0.092849,0.002616,0.007772,0.005236,0.007025,0.041311,-0.014993,...,0.007175,0.029099,-0.014174,0.021813,0.019759,-0.011484,-0.024164,0.001597,0.027427,0.041353


In [110]:
X = excess_returns.T.copy()

print("Shape of X (p x n):")
print(X.shape)

Shape of X (p x n):
(400, 26)


In [111]:
excess_returns.to_excel("step3_excess_returns.xlsx")
X.to_excel("step3_X_matrix.xlsx")

print("Saved as step3_excess_returns.xlsx")
print("Saved as step3_X_matrix.xlsx")

Saved as step3_excess_returns.xlsx
Saved as step3_X_matrix.xlsx


In [112]:
# Mean excess return for each asset (p x 1)
mu = X.mean(axis=1)

print("Shape of mu:")
print(mu.shape)

mu.head()

Shape of mu:
(400,)


Ticker
A       0.011736
AAPL    0.010701
ABBV    0.003673
ABNB    0.000467
ABT    -0.003087
dtype: float64

In [113]:
mu = mu.values.reshape(-1, 1)

print("Shape of mu (column):", mu.shape)

Shape of mu (column): (400, 1)


In [114]:
# Save expected excess returns (weekly and annual)
mu_weekly = mu.flatten()
mu_annual = 52 * mu_weekly

mu_df = pd.DataFrame({
    "Expected Excess Return (weekly)": mu_weekly,
    "Expected Excess Return (annual)": mu_annual
}, index=X.index)

mu_df.to_excel("step4_expected_excess_returns.xlsx")
print("Saved as step4_expected_excess_returns.xlsx")

Saved as step4_expected_excess_returns.xlsx


In [115]:
# number of weeks
n = X.shape[1]

# ones vector
ones = np.ones((1, n))

# De-mean
Y = X - mu @ ones

print("Shape of Y:")
print(Y.shape)

Shape of Y:
(400, 26)


In [116]:
S = (Y @ Y.T) / n

print("Shape of S:")
print(S.shape)

Shape of S:
(400, 400)


In [117]:
print("Trace of S:", np.trace(S))

Trace of S: 0.7751092466470767


In [118]:
# Eigenvalues and eigenvectors
eigvals, eigvecs = np.linalg.eigh(S)

In [119]:
# Largest eigenvalue
lambda2 = eigvals[-1]

# Corresponding eigenvector
h = eigvecs[:, -1].reshape(-1, 1)

print("lambda^2:", lambda2)
print("Shape of h:", h.shape)

lambda^2: 0.19082127135626617
Shape of h: (400, 1)


In [120]:
trace_S = np.trace(S)

ell2 = (trace_S - lambda2) / (n - 1)

print("ell^2:", ell2)

ell^2: 0.02337151901163242


In [121]:
p = 400
I = np.eye(p)

In [122]:
Sigma = (lambda2 - ell2) * (h @ h.T) + (n / p) * ell2 * I

print("Shape of Sigma:", Sigma.shape)

Shape of Sigma: (400, 400)


In [123]:
Sigma_inv = np.linalg.inv(Sigma)

In [124]:
p = 400
ones = np.ones((p, 1))

In [126]:
numerator = Sigma_inv @ ones
denominator = ones.T @ Sigma_inv @ ones

h_C = numerator / denominator

print("Shape of h_C:", h_C.shape)

weights_df = pd.DataFrame({
    "h_C": h_C.flatten()
}, index=X.index)

weights_df.to_excel("step5_portfolio_weights.xlsx")
print("Saved as step_portfolio_weights.xlsx")

f_C = (mu.T @ h_C).item()
print("Portfolio expected excess return (weekly):", f_C)

Shape of h_C: (400, 1)
Saved as step_portfolio_weights.xlsx
Portfolio expected excess return (weekly): 0.0017285251625939095


In [78]:
f_C = (mu.T @ h_C).item()

print("Portfolio expected excess return (weekly):", f_C)

Portfolio expected excess return (weekly): 0.00172853811381375


In [79]:
var_C = (h_C.T @ Sigma @ h_C).item()

print("Portfolio variance (weekly):", var_C)

Portfolio variance (weekly): 1.202123062073012e-05


In [80]:
std_C = np.sqrt(var_C)

print("Portfolio std dev (weekly):", std_C)

Portfolio std dev (weekly): 0.0034671646370961562


In [81]:
weeks = 52

f_C_annual = f_C * weeks
var_C_annual = var_C * weeks
std_C_annual = std_C * np.sqrt(weeks)

print("Annual expected excess return:", f_C_annual)
print("Annual variance:", var_C_annual)
print("Annual standard deviation:", std_C_annual)

Annual expected excess return: 0.08988398191831501
Annual variance: 0.0006251039922779662
Annual standard deviation: 0.02500207975905137


In [82]:
asset_variances = np.diag(Sigma)
asset_std = np.sqrt(asset_variances)

print("Shape of asset variances:", asset_variances.shape)
print("Shape of asset std devs:", asset_std.shape)

print("\nFirst 5 asset variances:")
print(asset_variances[:5])

print("\nFirst 5 asset std devs:")
print(asset_std[:5])

Shape of asset variances: (400,)
Shape of asset std devs: (400,)

First 5 asset variances:
[0.00237067 0.00207456 0.00156094 0.00242506 0.00155408]

First 5 asset std devs:
[0.04868956 0.04554738 0.03950877 0.04924492 0.03942186]


In [83]:
asset_variances_annual = asset_variances * weeks
asset_std_annual = asset_std * np.sqrt(weeks)

print("\nFirst 5 annualized variances:")
print(asset_variances_annual[:5])

print("\nFirst 5 annualized std devs:")
print(asset_std_annual[:5])


First 5 annualized variances:
[0.12327502 0.10787732 0.08116901 0.12610325 0.08081232]

First 5 annualized std devs:
[0.35110542 0.32844683 0.28490176 0.35511019 0.28427508]


In [84]:
# Convert to DataFrame for nicer output
asset_df = pd.DataFrame({
    "Variance (weekly)": asset_variances,
    "Std Dev (weekly)": asset_std,
    "Variance (annual)": asset_variances_annual,
    "Std Dev (annual)": asset_std_annual
})

asset_df.to_excel("step6_asset_stats.xlsx")

print("Saved asset statistics to step6_asset_stats.xlsx")

Saved asset statistics to step6_asset_stats.xlsx


In [127]:
portfolio_summary_df = pd.DataFrame({
    "Weekly": [f_C, var_C, std_C],
    "Annual": [f_C_annual, var_C_annual, std_C_annual]
}, index=[
    "Expected excess return",
    "Variance",
    "Standard deviation"
])

portfolio_summary_df.to_excel("portfolio_summary.xlsx")
print("Saved as portfolio_summary.xlsx")

Saved as portfolio_summary.xlsx
